In [ ]:
import os
import torch
from google.colab import drive
drive.mount('/content/drive')
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes -q
!pip install langchain langchain-community langchain-huggingface langchain-chroma chromadb sentence-transformers -q
print("bitti.")

In [ ]:
#4bit
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
FastLanguageModel.for_inference(model)
print("Model yüklendi.")

In [ ]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

VECTOR_DB_PATH = "/content/drive/MyDrive/hukuk-bot/data/vector_db"

embedding_function = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

if os.path.exists(VECTOR_DB_PATH):
    db = Chroma(persist_directory=VECTOR_DB_PATH, embedding_function=embedding_function)
    print(f"Veritabanı bağlandı: {VECTOR_DB_PATH}")
else:
    print("Hata: Vektör veritabanı bulunamadı.")

In [ ]:
def hukuk_asistani(soru):
    print(f"\nSoru: {soru}")
    docs = db.similarity_search(soru, k=6)

    if not docs:
        print("İlgili kaynak bulunamadı.")
        return

    context = "\n\n".join([doc.page_content for doc in docs])

    messages = [
        {
            "role": "system",
            "content": "Sen uzman bir Türk Hukuku asistanısın. Kullanıcının sorusunu SADECE aşağıdaki BAĞLAM metnine dayanarak cevapla. Cevabın profesyonel, hukuki ve %100 Türkçe olsun."
        },
        {
            "role": "user",
            "content": f"BAĞLAM:\n{context}\n\nSORU:\n{soru}"
        }
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    outputs = model.generate(
        input_ids,
        max_new_tokens=512,
        eos_token_id=terminators,
        do_sample=True,
        temperature=0.1,
        top_p=0.9,
    )

    response = outputs[0][input_ids.shape[-1]:]
    temiz_cevap = tokenizer.decode(response, skip_special_tokens=True)

    print(f"Cevap:\n{temiz_cevap}")
    print("-" * 50)

# Testler
hukuk_asistani("Kasten yaralama suçunun cezası nedir?")
hukuk_asistani("Kiracı kirayı ödemezse ev sahibi ne yapabilir?")
hukuk_asistani("Boşanma davası hangi durumlarda açılır?")